<a href="https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shadowwil-web/flyrank-machine-lerning-intern-subhasis/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*
* **Core Insight:** Content decay occurs in distinct structural patterns. High-traffic pages suffer from gradual engagement fatigue, while mid-tier pages experience ranking drop-offs.
* **Archetype → Action Mapping:**
  * **Archetype A: High-Visibility Fatigue** (High Impressions, Low/Declining CTR) → **Action:** *Metadata & Hook Optimization* (Rewrite title tags and meta descriptions).
  * **Archetype B: Keyword Slippage** (Declining Impressions, Moderate CTR) → **Action:** *Comprehensive Content Refresh* (Update facts, expand topical coverage, and add semantic subheadings).
  * **Archetype C: Stale Legacy Asset** (High Age, Low Impressions & CTR) → **Action:** *Consolidation or Pruning Review* (Evaluate for 301 redirection).
* **Reason Code Taxonomy:**
  * `RC_CTR_DECAY`: CTR dropped $\ge 25\%$ below historical baseline.
  * `RC_IMP_DROP`: 90-day impressions fell into bottom quartile.
  * `RC_AGE_STALE`: Content age exceeds 365 days without revision.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended Scope:** A decision-support tool designed to rank and prioritize a weekly batch of 50 candidate URLs for editorial review.
* **Model Boundaries & Known Limits:**
  * **No Causal Guarantee:** A high decay score indicates traffic vulnerability, not a guarantee that editing the page will instantly reclaim rankings.
  * **Blind to External Shocks:** The model cannot detect sudden macroeconomic shifts or algorithmic search engine updates.
  * **Non-Production Design:** Operates as an offline analytical scoring engine, not a real-time transactional system.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

* **Human Review Standard:** Every URL flagged in the top 50 queue must undergo manual editorial review prior to drafting changes.
* **The Strict "No-Go" List (What Must NOT Be Automated):**
  1. **Automated Content Overwrites:** Never allow scripts or LLMs to directly publish changes to live production URLs without human verification.
  2. **Seasonal / Cyclical Content:** Do not refresh holiday or seasonal campaign pages during their predictable off-season troughs.
  3. **Legal & Compliance Pages:** Exclude terms of service, privacy policies, and regulated disclaimer pages from refresh queues.
  4. **Newly Published Content ($<90$ days old):** Exclude new URLs while they are still in their initial search ranking discovery phase.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

* **Monitoring & Drift Triggers:**
  * **Feature Drift:** If the median 90-day impression distribution shifts by $>20\%$ month-over-month, recompute baseline normalizations.
  * **Performance Decay:** If Precision@50 on human-validated review acceptance falls below $65\%$, trigger a full model retraining cycle.
  * **Quarterly Cadence:** Re-fit Random Forest hyperparameters quarterly using the latest 90-day observation window.
* **Cost vs. Value Framework:**
  * *Cost of False Positive:* $\approx 15\text{ minutes}$ of an editor’s review time.
  * *Cost of False Negative:* Missed recovery of valuable organic search visits over subsequent quarters.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. SETUP DIRECTORY STRUCTURE
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/metrics", exist_ok=True)

# 2. AUTHENTICATE & LOAD DATA
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train[:100000]",
    token=hf_token
)
df = dataset.to_pandas()

df_active = df[df['is_available'] == True].copy() if 'is_available' in df.columns else df.copy()

target_col = df_active.select_dtypes(include=[np.number]).columns[0]
df_active['target_declining'] = (df_active[target_col] < df_active[target_col].median()).astype(int)

imp_col = 'impressions_90d' if 'impressions_90d' in df_active.columns else target_col
age_col = 'content_age_days' if 'content_age_days' in df_active.columns else df_active.select_dtypes(include=[np.number]).columns[1]
click_col = 'clicks_90d' if 'clicks_90d' in df_active.columns else df_active.select_dtypes(include=[np.number]).columns[2]

df_active['ctr'] = df_active[click_col] / (df_active[imp_col] + 1)
feature_cols = [imp_col, age_col, click_col, 'ctr']
X = df_active[feature_cols].fillna(0)
y = df_active['target_declining']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

df_active['decay_probability'] = rf_model.predict_proba(X)[:, 1]

# Export ranked queue CSV
ranked_queue = df_active.sort_values('decay_probability', ascending=False).head(50)
output_cols = [c for c in ['page_id', 'url', imp_col, age_col, 'ctr', 'decay_probability'] if c in df_active.columns]
ranked_queue[output_cols].to_csv("work/outputs/ranked_refresh_queue.csv", index=False)
print("Exported: work/outputs/ranked_refresh_queue.csv")

# Export metrics JSON
metrics_summary = {
    "model_type": "RandomForestClassifier",
    "top_50_mean_decay_prob": float(ranked_queue['decay_probability'].mean())
}
with open("work/metrics/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=4)
print("Exported: work/metrics/w07_playbook_metrics.json")

# Export figure
plt.figure(figsize=(8, 4))
plt.hist(df_active['decay_probability'], bins=30, color='#2b5c8f', edgecolor='black', alpha=0.7)
plt.title('Distribution of Content Decay Scores')
plt.xlabel('Predicted Decay Probability')
plt.ylabel('URL Count')
plt.tight_layout()
plt.savefig("work/figures/decay_score_distribution.png", dpi=300)
plt.close()
print("Exported: work/figures/decay_score_distribution.png")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 2.62MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Exported: work/outputs/ranked_refresh_queue.csv
Exported: work/metrics/w07_playbook_metrics.json
Exported: work/figures/decay_score_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.